# Phase Estimation Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Quantum Phase Estimation" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [2]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import QPU, Qubits


## Problem 1.  Find eigenvalues of the S gate

Since the $S$ gate is diagonal, it's easy to realize that its eigenvectors are the basis vectors $\ket{0}$ and $\ket{1}$.

To find the corresponding eigenvalues, you need to solve the two equations:
- $S\ket{0} = \lambda \ket{0}$, which gives you $\lambda = 1$.
- $S\ket{1} = \lambda \ket{1}$, which gives you $\lambda = i$.

In [ ]:
def eigenvalues_s() -> list[complex]:
    return [1, 1j]

## Problem 2. Find eigenvalues of the X gate

Since the $X$ gate is self-adjoint, you know that its eigenvalues can only be $+1$ and $-1$.
Now, you need to find the eigenvectors that correspond to these eigenvalues. 
To do this, you need to solve the two equations:
- $X \begin{bmatrix} v_0 \\ v_1 \end{bmatrix} = \begin{bmatrix} v_0 \\ v_1 \end{bmatrix}$, which gives you $v_0 = v_1$.
- $X \begin{bmatrix} v_0 \\ v_1 \end{bmatrix} = -\begin{bmatrix} v_0 \\ v_1 \end{bmatrix}$, which gives you $v_0 = -v_1$.

One of the eigenvectors should consist of two equal elements, and the other - of two elements with equal absolute values but opposite signs.

In [ ]:
def eigenvectors_x() -> list[list[float]]:
    return [[1, 1], [1, -1]]

## Problem 3. Is given state an eigenvector of the gate?

A quantum state is an eigenstate of a quantum gate if applying that gate to that state doesn't change it, other than multiply it by a global phase. This means that your solution should start by preparing the state $\ket{\psi}$ and applying the unitary $U$ to it. How can you check that the state after that is still $\ket{\psi}$ (up to a global phase)?

To do this, you need to have access to the state vector of the simulated program before and after applying the unitary $U$. You can fetch it using the `pull_state` method of the Qubits class.

You'll need to get two state vectors: first, the given state $\ket{psi}$, and second, the state $U\ket{psi}$.
Then, you need to check whether these vectors are the same up to a global phase. You can do that using the function `fidelity` from the namespace `psiqworkbench.utils.numpy_utils`.

In [ ]:
from math import isclose
from psiqdk.workbench.utils.numpy_utils import fidelity

def is_eigenvector(U: callable, P: callable) -> bool:
    qpu = QPU(num_qubits=1)
    x = Qubits(1, "x", qpu)

    # Prepare |ψ⟩ and fetch its state vector
    P(x)
    psi = x.pull_state()
    
    # Apply unitary U and fetch state vector U|ψ⟩
    U(x)
    u_psi = x.pull_state()
    x.release()

    # Check that two vectors are only different by an absolute phase
    return isclose(abs(fidelity(psi, u_psi)), 1)

## Problem 4. One-bit phase estimation

What happens if you apply the unitary $U$ to the state $\ket{\psi}$? You end up in one of the states $\ket{\psi}$ or $-\ket{\psi}$, depending on whether the eigenvalue is $+1$ or $-1$. You need to distinguish these two scenarios, but they differ only by a global phase, so you can't do this using just one qubit.

However, if you use the fact that you have access to controlled variant of $U$ and can allocate more than one qubit, you can use a variant of the phase kickback trick to distinguish these scenarios.

If you apply a controlled $U$ gate to two qubits: the control qubit in the $\ket{+}$ state and the target qubit in the state $\ket{\psi}$, you'll get the following state:

$$CU \ket{+}\ket{\psi} = \tfrac1{\sqrt2}(CU \ket{0}\ket{\psi} + CU \ket{1}\ket{\psi}) = \tfrac1{\sqrt2}(\ket{0}\ket{\psi} + \ket{1}U\ket{\psi}) = $$
$$= \frac1{\sqrt2}(\ket{0}\ket{\psi} + \ket{1}\lambda\ket{\psi}) = 
\begin{cases}
\ket{+}\ket{\psi} \textrm{ if } \lambda = 1 \\ 
\ket{-}\ket{\psi} \textrm{ if } \lambda = -1
\end{cases}$$

You only need to measure the control qubit in the Hadamard basis to figure out whether its state is $\ket{+}$ or $\ket{-}$, and you'll be able to tell the value of the eigenvalue $\lambda$.

In [ ]:
def one_bit_phase_estimation(U: callable, P: callable) -> int:
    qpu = QPU(num_qubits=2)
    control = Qubits(1, "control", qpu)
    eigenstate = Qubits(1, "eigenstate", qpu)
    
    # Prepare state |+⟩|ψ⟩
    control.had()
    P(eigenstate)

    # Apply controlled variant of the unitary
    U(eigenstate, cond=control)

    # Measure control qubit in Hadamard basis
    control.had()
    res = control.read()

    control.release()
    eigenstate.release()

    # Convert measured eigenphase to eigenvalue
    return 1 if res == 0 else -1

## Problem 5. Implement QPE algorithm

To solve this exercise, you need to follow the QPE algorithm as described in this lesson.

1. Allocate two qubit registers, $n$-qubit `phase_register` and single-qubit `eigenstate`.
2. Prepare the initial state of the algorithm: `eigenstate` in the state $\ket{\psi}$ using the unitary $P$ and `phase_register` in the state that is an even superposition of all basis states using Hadamard gates.
3. Apply the controlled $U$ gates using two `for` loops. The outer loop will iterate over the qubits of `phase_register`, from the first qubit storing the least significant digit to the last one storing the most significant one. The inner loop will apply controlled $U$ $2^k$ times, where $k$ is the variable used as the outer loop counter.
4. Apply the adjoint QFT. Here, you can use the library operation `QFT`, calling its adjoint by setting `dagger=True`.
5. Measure the qubits in `phase_register` and convert the result to an integer. You also need to reset the qubit used as the eigenstate before releasing it.

In [ ]:
def phase_estimation(U: callable, P: callable, n: int) -> int:
    from psiqdk.algorithms import QFT
    qft = QFT()
    
    qpu = QPU(num_qubits=n + 1)
    phase_register = Qubits(n, "phase_register", qpu)
    eigenstate = Qubits(1, "eigenstate", qpu)

    # Prepare the initial state
    P(eigenstate)
    phase_register.had()

    # Apply controlled-U ladder
    for k in range(n):
        for _ in range(2 ** k):
            U(eigenstate, cond=phase_register[k])

    # Apply inverse QFT
    qft.compute(phase_register, dagger=True)

    # Read out the resulting phase
    result = phase_register.read()

    eigenstate.release()
    phase_register.release()
    return result 

> Copyright (c) 2026 PsiQuantum